In [ ]:
import pika
import time

def callback(ch, method, properties, body):
        print(f"Msg recieved: {body.decode()}")


host = '149.62.71.186'
queue = 'test_queue_2'
credentials = pika.PlainCredentials('martin', 'martin00')
parameters =  pika.ConnectionParameters(host=host, credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()
#channel.queue_purge(queue=queue) #če želite izbrisati queue
channel.queue_declare(queue=queue)

#auto ack = True!
channel.basic_consume(queue=queue, on_message_callback=callback, auto_ack=True)

for i in range(0,10):
    time.sleep(0.1)
    channel.basic_publish(exchange='', routing_key=queue, body=f'Msg n.{i+1}ššđđ')

#tole bo bralo podatke, a blokiralo program
channel.start_consuming()

In [ ]:
#if you forget to close, you're gonna have problems!
connection.close()

In [ ]:
import pika
import time

#the definition of callback function must strictly be such!
def callback(ch, method, properties, body):
    print(f"Msg recieved: {body.decode()}")
    # Simulate processing (replace with your actual processing logic)
    try:
        # Process the message (e.g., print, store in database)
        print(f"Processing message: {body}")
        # Acknowledge the message after successful processing
        ch.basic_ack(delivery_tag=method.delivery_tag)
    except Exception as e:
        print(f"Error processing message: {body}")
        # Optionally redeliver the message using nack with requeue=True
        # ch.basic_nack(delivery_tag=method.delivery_tag, requeue=True)

host = '149.62.71.186'
queue = 'test_queue_2023'
credentials = pika.PlainCredentials('martin', 'martin00')
parameters = pika.ConnectionParameters(host=host, credentials=credentials)
connection = pika.BlockingConnection(parameters)
channel = connection.channel()
channel.queue_declare(queue=queue)
#auto ack = False !!
channel.basic_consume(queue=queue, on_message_callback=callback, auto_ack=False)

for i in range(0, 10):
    time.sleep(1)
    channel.basic_publish(exchange='', routing_key=queue, body=f'my msg n.{i+1}')
    print(f'my msg n.{i+1}')

channel.start_consuming()

In [ ]:
connection.close()

The code you provided defines a consumer program that listens for messages on a RabbitMQ queue named "test_queue" and prints them when received. Here's a breakdown of the code:

**Imports:**

- `import pika`: Imports the `pika` library for interacting with RabbitMQ.
- `import time`: Imports the `time` library for using the `sleep` function.

**Callback Function:**

- `def callback(ch, method, properties, body):`: This defines a callback function named `callback`. This function is called whenever a message is received on the subscribed queue.
    - `ch`: This parameter represents the channel object.
    - `method`: This parameter contains information about the received message, like delivery tags and routing keys.
    - `properties`: This parameter holds message properties like headers and content type.
    - `body`: This parameter contains the actual message content (the payload).
    - Inside the function, it prints the received message body using f-strings.

**RabbitMQ Interaction:**

- `host = '149.62.71.186'`: Defines the RabbitMQ server address.
- `queue = 'test_queue'`: Defines the queue name to listen on.
- `credentials = pika.PlainCredentials('martin', 'martin00')`: Creates credentials for authentication (update with your credentials).
- `parameters = pika.ConnectionParameters(host=host, credentials=credentials)`: Creates connection parameters.
- `connection = pika.BlockingConnection(parameters)`: Establishes a blocking connection to the RabbitMQ server.
- `channel = connection.channel()`: Creates a channel on the connection.
- `channel.queue_declare(queue=queue)`: Declares the "test_queue" if it doesn't already exist.
- `channel.basic_consume(queue=queue, on_message_callback=callback, auto_ack=True)`: Sets up message consumption on the queue.
    - `queue`: The queue name to listen on ("test_queue").
    - `on_message_callback`: The callback function to be called for each received message (set to `callback`).
    - `auto_ack`: Acknowledges receiving the message automatically after processing (set to `True` here).

**Publishing Messages (Simulation):**

- The `for` loop iterates 10 times (0 to 9).
- Inside the loop:
    - `time.sleep(1)`: Pauses the program execution for 1 second between each message.
    - `channel.basic_publish(exchange='', routing_key=queue, body=f'Msg n.{i+1}')`: Publishes a message with the content "Msg n.{i+1}" (where n is the message number) to the "test_queue". The exchange is set to an empty string for direct queue delivery.

**Starting Consumption and Closing Connection:**

- `channel.start_consuming()`: Starts consuming messages from the queue (waits for messages and calls the callback function).
- `connection.close()`: Closes the connection to the RabbitMQ server after consumption is finished.

**Note:**

- This code simulates sending messages by using a loop with `time.sleep`. In a real scenario, messages would likely come from another source.
- Be mindful of storing credentials directly in code. Consider environment variables or configuration files for better security practices.